# 3. Pore analysis with Zeo++ (Windows edition)

Language: **English** | [中文](../cn/03_zeopp_pore_analysis.ipynb)

This native-Windows edition locates the TAPB–TFB CIF generated by the Windows edition of Notebook 1, runs cofkit's Zeo++ analysis wrapper from PowerShell, and prints human-readable pore information without requiring WSL. The equivalent `analyze_zeopp_pore_properties` Python workflow is included as commented code.

## Tutorial series

1. [Environment and first build](01_environment_and_first_build.ipynb)
2. [Topology, connectivity, and linkage examples](02_topologies_connectivities_and_linkages.ipynb)
3. **Pore analysis with Zeo++** (this notebook)

Prerequisites: run the Windows edition of Notebook 1 successfully and have a Windows-runnable Zeo++ 0.3 `network.exe` or `network` executable. The `cofkit` Python package itself works without Zeo++; only this analysis requires the external binary.

## Locate Notebook 1's CIF

The build pipeline groups CIFs by coarse validation class, so the exact subdirectory can vary. This search deliberately looks through all classes and fails with a useful message if Notebook 1 has not produced a CIF.

In [ ]:
%%script powershell -NoProfile
$ErrorActionPreference = "Stop"
$root = (& git rev-parse --show-toplevel).Trim()
if ($LASTEXITCODE -ne 0) { throw "git rev-parse failed with exit code $LASTEXITCODE." }
Set-Location -LiteralPath $root
$cif = Get-ChildItem -LiteralPath "out/tutorial_01_first_build/cifs" -Recurse -File -Filter "tapb__tfb__hcb.cif" -ErrorAction SilentlyContinue | Select-Object -First 1
if ($null -eq $cif) { throw "No TAPB-TFB CIF was found. Run the Windows edition of Notebook 1 before continuing." }
Write-Host "Using CIF: $($cif.FullName)"
Get-Content -LiteralPath $cif.FullName | Select-Object -First 18

In [ ]:
# Python equivalent (commented out): locate Notebook 1's CIF.
# from pathlib import Path
# matches = sorted(Path("out/tutorial_01_first_build/cifs").rglob("tapb__tfb__hcb.cif"))
# if not matches:
#     raise FileNotFoundError("Run Notebook 1 before continuing.")
# cif_path = matches[0]
# print(cif_path)

## Configure Zeo++ on Windows

Set `COFKIT_ZEOPP_PATH` to a Windows-runnable Zeo++ executable before launching Jupyter:

```powershell
$env:COFKIT_ZEOPP_PATH = "C:\tools\zeo++-0.3\network.exe"
jupyter lab
```

Upstream Zeo++ 0.3 documents Windows compilation through Cygwin and GNU Make. Compile it outside this notebook or use an instructor-provided binary, then ensure any required Cygwin runtime directory is on `PATH`. The next cell validates the configured file; it intentionally does not attempt to install Cygwin or a compiler inside a restricted Windows container.

In [ ]:
%%script powershell -NoProfile
$ErrorActionPreference = "Stop"
if ([string]::IsNullOrWhiteSpace($env:COFKIT_ZEOPP_PATH)) {
  throw "COFKIT_ZEOPP_PATH is not set. Point it to network.exe (or network) before launching Jupyter."
}
if (-not (Test-Path -LiteralPath $env:COFKIT_ZEOPP_PATH -PathType Leaf)) {
  throw "Zeo++ executable does not exist: $env:COFKIT_ZEOPP_PATH"
}
$zeoppPath = (Resolve-Path -LiteralPath $env:COFKIT_ZEOPP_PATH).Path
Write-Host "Using configured Zeo++: $zeoppPath"

In [ ]:
# There is intentionally no Python API equivalent for configuring or compiling Zeo++ on Windows.
# Zeo++ is an external C++ program; cofkit's Python API begins at the analysis step below.

## Run point-probe and finite-probe analysis

The baseline reports pore-limiting sphere sizes plus point-probe surface area and volume. The repeated `--probe-radius` values add accessibility-aware scans at illustrative radii of 1.20 Å and 1.86 Å. Adjust these radii to match the adsorbate model used in your own study.

The command intentionally omits `--json`: the terminal output is the concise human-readable report. A complete machine-readable `zeopp_report.json` is still written to disk.

In [ ]:
%%script powershell -NoProfile
$ErrorActionPreference = "Stop"
$root = (& git rev-parse --show-toplevel).Trim()
if ($LASTEXITCODE -ne 0) { throw "git rev-parse failed with exit code $LASTEXITCODE." }
Set-Location -LiteralPath $root
$cif = Get-ChildItem -LiteralPath "out/tutorial_01_first_build/cifs" -Recurse -File -Filter "tapb__tfb__hcb.cif" -ErrorAction SilentlyContinue | Select-Object -First 1
if ($null -eq $cif) { throw "Run the Windows edition of Notebook 1 first." }
if ([string]::IsNullOrWhiteSpace($env:COFKIT_ZEOPP_PATH) -or -not (Test-Path -LiteralPath $env:COFKIT_ZEOPP_PATH -PathType Leaf)) {
  throw "Set COFKIT_ZEOPP_PATH to a valid Windows-runnable Zeo++ executable."
}
$zeoppPath = (Resolve-Path -LiteralPath $env:COFKIT_ZEOPP_PATH).Path
$cofkitArguments = @(
  "analyze", "zeopp", $cif.FullName,
  "--zeopp-path", $zeoppPath,
  "--output-dir", "out/tutorial_03_zeopp",
  "--probe-radius", "1.20",
  "--probe-radius", "1.86",
  "--surface-samples-per-atom", "250",
  "--volume-samples-total", "5000"
)
& conda run -n cofkit cofkit @cofkitArguments
if ($LASTEXITCODE -ne 0) { throw "Zeo++ analysis failed with exit code $LASTEXITCODE." }

In [ ]:
# Python API equivalent (commented out), including a human-readable report.
# import os
# from pathlib import Path
# from cofkit import analyze_zeopp_pore_properties
#
# cif_path = next(Path("out/tutorial_01_first_build/cifs").rglob("tapb__tfb__hcb.cif"))
# zeopp_binary = os.environ["COFKIT_ZEOPP_PATH"]
# result = analyze_zeopp_pore_properties(
#     cif_path,
#     output_dir="out/tutorial_03_zeopp_python",
#     probe_radii=(1.20, 1.86),
#     surface_samples_per_atom=250,
#     volume_samples_total=5000,
#     zeopp_path=zeopp_binary,
# )
# basic = result.baseline.basic_pore_properties
# channels = result.baseline.point_probe_channels
# surface = result.baseline.point_probe_surface_area
# volume = result.baseline.point_probe_volume
# print(f"Input CIF: {result.input_cif}")
# print(f"Largest included sphere: {basic.largest_included_sphere:.3f} Å")
# print(f"Largest free sphere: {basic.largest_free_sphere:.3f} Å")
# print("Largest included sphere along a free path: "
#       f"{basic.largest_included_sphere_along_free_path:.3f} Å")
# print(f"Point-probe channels / pockets: {channels.n_channels} / {channels.n_pockets}")
# print(f"Point-probe accessible surface area: {surface.accessible_surface_area_a2:.3f} Å²")
# print(f"Point-probe accessible volume: {volume.accessible_volume_a3:.3f} Å³")
# for scan in result.probe_scans:
#     print(f"Probe radius {scan.settings.probe_radius:.2f} Å: {scan.status}")
#     if scan.surface_area is not None and scan.volume is not None:
#         print(f"  accessible surface area = {scan.surface_area.accessible_surface_area_a2:.3f} Å²")
#         print(f"  accessible volume fraction = {scan.volume.accessible_volume_fraction:.5f}")
#     if scan.accessibility is not None:
#         print(f"  accessible Voronoi-node fraction = {scan.accessibility.accessible_fraction}")
# print(f"Full report: {result.report_path}")

## How to read the main values

| Report field | Practical meaning |
|---|---|
| Largest included sphere | Diameter of the largest sphere that fits somewhere in the pore network |
| Largest free sphere | Diameter of the largest sphere that can traverse the periodic pore network |
| Largest included sphere along free path | Largest cavity encountered along a traversable path |
| Accessible surface area | Surface reachable by the selected probe under Zeo++'s geometric model |
| Accessible volume / fraction | Probe-accessible pore volume, either absolute or relative to the unit cell |
| Channels and pockets | Connected transport pathways versus isolated accessible regions |

All sphere sizes and probe settings in the report are in ångström-based units. Surface-area and volume estimates use Monte Carlo sampling, so increase the sample counts for production convergence studies.

## Inspect the retained report and raw outputs

cofkit preserves Zeo++ input/output artifacts and logs so the parsed report remains auditable.

In [ ]:
%%script powershell -NoProfile
$ErrorActionPreference = "Stop"
$root = (& git rev-parse --show-toplevel).Trim()
if ($LASTEXITCODE -ne 0) { throw "git rev-parse failed with exit code $LASTEXITCODE." }
Set-Location -LiteralPath $root
Write-Host "--- generated Zeo++ files ---"
Get-ChildItem -LiteralPath "out/tutorial_03_zeopp" -Recurse -File | Sort-Object FullName | ForEach-Object { $_.FullName }
Write-Host "--- zeopp_report.json (first 140 lines) ---"
Get-Content -LiteralPath "out/tutorial_03_zeopp/zeopp_report.json" | Select-Object -First 140

In [ ]:
# Python equivalent (commented out): read the persisted report.
# import json
# from pathlib import Path
# report_path = Path("out/tutorial_03_zeopp/zeopp_report.json")
# report = json.loads(report_path.read_text())
# print(report["baseline"]["basic_pore_properties"])
# for scan in report["probe_scans"]:
#     print(scan["settings"]["probe_radius"], scan["status"])

## Before using the numbers

Pore descriptors are only as meaningful as the input geometry and atomic radii model. Treat this generated CIF as a tutorial input; for research calculations, optimize and validate the framework first, document the Zeo++ version and radii definitions, and check sampling convergence. A failed finite-probe scan is recorded in the JSON report without discarding successful baseline results.